In [1]:
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp
import time
from multiprocessing import Process
import multiprocessing
from random import randint
from itertools import product
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Parameters
pkt_prob = 0.5
e_prob = 0.5
B_max = 2
rmax = 1
M = 2

B_vals   = np.arange(0, B_max + 1, 1)
rem_vals = np.arange(0, rmax + 0.5, 0.5)

all_states = []
for B in product(B_vals, repeat=M):                 # B1..BM
    for rem in product(rem_vals, repeat=M):         # rem1..remM
        state = (*B, *rem)
        all_states.append(state)
all_states = np.array(all_states)
print(all_states.shape)

(81, 4)


In [3]:
new_state = all_states[:: 1]
print(new_state.shape)

(81, 4)


In [4]:
all_states = new_state

In [5]:
def one_slot_greedy_T_Mu(state):
    B   = state[0:M]
    rem = state[M:2*M]
    rho = [cp.Variable(nonneg=True) for _ in range(M)]
    P   = [cp.Variable(nonneg=True) for _ in range(M)]
    obj = 0
    for i in range(M):
        obj += cp.exp(-(rmax - (rem[i] - rho[i])))
    log2 = cp.inv_pos(cp.log(2))
    cons = []
    for i in range(M):
        cons += [
            P[i]   <= B[i],
            rho[i] <= rem[i],
            rho[i] <= cp.log(1 + P[i]) * log2
            ]
    # --- Full MAC subset constraints ---
    from itertools import combinations
    users = list(range(M))
    for k in range(2, M+1):
        for S in combinations(users, k):
            sum_rho = sum(rho[i] for i in S)
            sum_pow = sum(P[i] for i in S)
            cons.append(sum_rho <= cp.log(1 + sum_pow) * log2)
    # --- Solve ---
    prob = cp.Problem(cp.Minimize(obj), cons)
    # return prob.solve(verbose=False)
    try:
       optimal_val =  prob.solve(solver=cp.ECOS, warm_start=True, verbose=False)
    except:
        optimal_val = prob.solve(solver=cp.SCS, eps=1e-4, max_iters=5000, verbose=False)
    return optimal_val
    

In [6]:
V = []
for state in all_states:
        V.append(one_slot_greedy_T_Mu(state))
V = np.array(V)
V= np.abs(V)
print(V.shape)

(81,)


In [7]:
import cvxpy as cp
import numpy as np

def quad_fit(all_states, Vy):
    batch_size=500
    X = all_states
    y = Vy.reshape(-1, 1)
    N, n = X.shape
    
    P = cp.Variable((n, n), symmetric=True)
    q = cp.Variable((n, 1))
    r = cp.Variable()
    
    objective_expr = 0
    constraints = [P >> 0]
    
    for k in range(0, N, batch_size):
        Xb = X[k:k+batch_size]
        yb = y[k:k+batch_size]
        
        quad = cp.sum(cp.multiply(Xb @ P, Xb), axis=1, keepdims=True)
        lin  = Xb @ q
        preds = quad + lin + r
        
        objective_expr += cp.sum_squares(preds - yb)
        constraints.append(preds >= 1e-8)
    
    prob = cp.Problem(cp.Minimize(objective_expr), constraints)
    prob.solve(solver=cp.SCS, eps=1e-8)
    # prob.solve(solver=cp.ECOS, warm_start=True, verbose=False)
    
    
    P_opt = P.value
    q_opt = q.value.flatten()
    r_opt = float(r.value)
    
    return P_opt, q_opt, r_opt, 0

In [8]:
def one_slot_greedy_2(state, P_opt, q_opt, r_opt, gamma, send_end):
    B   = state[0:M]
    rem = state[M:2*M]

    rho = [cp.Variable(nonneg=True) for _ in range(M)]
    P   = [cp.Variable(nonneg=True) for _ in range(M)]
    
    obj_fn = 0
    for i in range(M):
        obj_fn +=  cp.exp(-(rmax - (rem[i] - rho[i])))
    ####################################################
    B_end = [B[i] - P[i] for i in range(M)]
    B_ch_arr = [[B_end[i], B_end[i] + 1] for i in range(M)]
    r_ch_arr = [[rem[i] - rho[i], rmax] for i in range(M)]

    all_possible_next_states = []
    for combo in product(*B_ch_arr, *r_ch_arr):
        ns = list(combo)
        all_possible_next_states.append(ns)
    #####################################################
    Vz = []
    for next_state in all_possible_next_states:
        ns_vec = cp.hstack(next_state)
        value = cp.quad_form(ns_vec, P_opt) + ns_vec @ q_opt + r_opt
        Vz.append(value)
    Vz = cp.hstack(Vz)
    #############################################
    B_ch_pr = np.array([1 - e_prob, e_prob])
    r_ch_pr = np.array([1 - pkt_prob, pkt_prob])

    next_state_prob = []
    for combo in product(*([B_ch_pr]*M), *([r_ch_pr]*M)):
        pr = 1
        for x in combo:
            pr *= x
        next_state_prob.append(pr)
    next_state_prob = np.array(next_state_prob)
    next_state_prob = cp.hstack(np.array(next_state_prob))
    ####################################################################
    Expected_V = gamma * (Vz @ next_state_prob)
    obj_fn += Expected_V
    constraints = []
    for i in range(M):
        constraints += [
            P[i]   >= 0,
            rho[i] >= 0,
            P[i]   <= B[i],
            rho[i] <= rem[i]
        ]

    log2 = cp.inv_pos(cp.log(2))
    # Single-user MAC bounds
    for i in range(M):
        constraints.append(
            rho[i] <= cp.log(1 + P[i]) * log2)
    # Full MAC subset constraints
    from itertools import combinations
    users = list(range(M))
    for k in range(2, M+1):
        for S in combinations(users, k):
            sum_rho = sum(rho[i] for i in S)
            sum_pow = sum(P[i] for i in S)
            constraints.append(
                sum_rho <= cp.log(1 + sum_pow) * log2)
    objective = cp.Minimize(obj_fn)
    prob = cp.Problem(objective, constraints)
    # optimal_val = prob.solve(verbose=False)
    try:
       optimal_val =  prob.solve(solver=cp.ECOS, warm_start=True, verbose=False)
    except:
        optimal_val = prob.solve(solver=cp.SCS, eps=1e-4, max_iters=5000, verbose=False)
    send_end.send(optimal_val)

In [9]:
T = 6
stp = 1
import time
while(stp < T):
    t11 = time.time()
    print('stp', stp)
    P_opt, q_opt, r_opt, flag = quad_fit(all_states, V)
    def make_positive_semidefinite(matrix, tolerance=1e-10):
        matrix = (matrix + matrix.T) / 2
        eigenvalues, eigenvectors = np.linalg.eigh(matrix)
        eigenvalues[eigenvalues < tolerance] = 0
        perturbed_matrix = eigenvectors @ np.diag(eigenvalues) @ eigenvectors.T
        perturbed_matrix = (perturbed_matrix + perturbed_matrix.T) / 2
        return perturbed_matrix
    # Check if the perturbed matrix is PSD
    def is_positive_semidefinite(matrix):
        eigenvalues = np.linalg.eigvals(matrix)
        return np.all(eigenvalues >= 0)    
    if flag == 1:
        psd_matrix = make_positive_semidefinite(P_opt)
        P_opt = psd_matrix
        # print("\nPerturbed matrix:")
        # print(psd_matrix)
        if is_positive_semidefinite(psd_matrix):
            print("\nThe perturbed matrix is positive semidefinite.")
        else:
            print("\nThe perturbed matrix is not positive semidefinite.")
    # policy_run(100,P_opt, q_opt, r_opt)
    psd_matrix = make_positive_semidefinite(P_opt)
    P_opt = psd_matrix
    # print(P_opt.shape, q_opt.shape)
    V_next = np.array([])
    z = 9
    q = 0
    while(z <= len(all_states)):
        t2 = time.time()
        jobs = []
        pipe_list = []
        for state in all_states[q:z]:
            recv_end, send_end = multiprocessing.Pipe(False)
            p = Process(target=one_slot_greedy_2, args=(state, P_opt, q_opt, r_opt, 0.99, send_end))
            jobs.append(p)
            pipe_list.append(recv_end)
        for process in jobs:
            process.start()
        for process in jobs:
            process.join()
        V_next_1 = np.array([x.recv() for x in pipe_list])
        V_next = np.concatenate((V_next, V_next_1))
        q = z
        z+= 9
        if z%81 == 0:
            print('z', z)
        # print(time.time() - t2)
    V_next = np.abs(V_next)
    V  = V_next
    # print(V, V.shape)
    print('-----------------------')
    # print(time.time() - t11)
    stp+= 1
print('value iteration done', pkt_prob, e_prob)

stp 1


z 81
-----------------------
stp 2
z 81
-----------------------
stp 3
z 81
-----------------------
stp 4
z 81
-----------------------
stp 5
z 81
-----------------------
value iteration done 0.5 0.5


In [10]:
# def one_slot_greedy_33(state, P_opt, q_opt, r_opt, gamma=0.99):
#     B   = state[0:M]
#     rem = state[M:2*M]
    
#     rho = [cp.Variable(nonneg=True) for _ in range(M)]
#     P   = [cp.Variable(nonneg=True) for _ in range(M)]
    
#     obj_fn = 0
#     for i in range(M):
#         obj_fn += cp.exp(-(rmax - (rem[i] - rho[i])))
#     ####################################################
#     B_end = [B[i] - P[i] for i in range(M)]
#     B_ch_arr = [[B_end[i], B_end[i] + 1] for i in range(M)]
#     r_ch_arr = [[rem[i] - rho[i], rmax] for i in range(M)]

#     all_possible_next_states = []
#     for combo in product(*B_ch_arr, *r_ch_arr):
#         ns = list(combo)
#         all_possible_next_states.append(ns)
#     #####################################################
#     Vz = []
#     for next_state in all_possible_next_states:
#         ns_vec = cp.hstack(next_state)
#         value = cp.quad_form(ns_vec, P_opt) + ns_vec @ q_opt + r_opt
#         Vz.append(value)
#     Vz = cp.hstack(Vz)
#     #############################################
#     B_ch_pr = np.array([1 - e_prob, e_prob])
#     r_ch_pr = np.array([1 - pkt_prob, pkt_prob])
    
#     next_state_prob = []
#     for combo in product(*([B_ch_pr]*M), *([r_ch_pr]*M)):
#         pr = 1
#         for x in combo:
#             pr *= x
#         next_state_prob.append(pr)
#     next_state_prob = np.array(next_state_prob)
#     next_state_prob = cp.hstack(np.array(next_state_prob))
#     ####################################################################
#     Expected_V = gamma * (Vz @ next_state_prob)
#     obj_fn += Expected_V
#     constraints = []
#     for i in range(M):
#         constraints += [
#             P[i]   >= 0,
#             rho[i] >= 0,
#             P[i]   <= B[i],
#             rho[i] <= rem[i] ]
#     log2 = cp.inv_pos(cp.log(2))
#     # Single-user bounds
#     for i in range(M):
#         constraints.append(
#             rho[i] <= cp.log(1 +P[i]) * log2 )
#     # Full MAC region (all subsets)
#     from itertools import combinations
#     users = list(range(M))
#     for k in range(2, M+1):
#         for S in combinations(users, k):
#             sum_rho = sum(rho[i] for i in S)
#             sum_pow = sum(P[i] for i in S)
#             constraints.append(
#                 sum_rho <= cp.log(1 + sum_pow) * log2)
#     objective = cp.Minimize(obj_fn)
#     prob = cp.Problem(objective, constraints)
#     # optimal_val = prob.solve(verbose=False)
#     optimal_val = prob.solve(solver=cp.ECOS, warm_start=True, verbose=False)
#     P_val   = [Pi.value for Pi in P]
#     rho_val = [ri.value for ri in rho]
#     return P_val, rho_val, optimal_val

In [11]:
# def policy_run(T_horizon, P_opt, q_opt, r_opt):
#     tot_dist = 0
#     B_prev  = np.ones(M)      # initial battery
#     rem_prev = np.zeros(M)    # initial remaining bits
    
#     for t in range(T_horizon):
       
#         pkt_rand = np.random.choice([1,0], size=M, p=[pkt_prob, 1-pkt_prob])
#         E_rand   = np.random.choice([1,0], size=M, p=[e_prob, 1-e_prob])

#         rem_start = np.zeros(M)
    
#         for i in range(M):
#             if pkt_rand[i] == 1:
#                 rem_start[i] = rmax
#             else:
#                 rem_start[i] = rem_prev[i]
#         B_start = np.minimum(B_prev + E_rand, B_max)
#         state_slot = np.concatenate([B_start, rem_start])

#         P_val, rho_val, VT = one_slot_greedy_33(state_slot, P_opt, q_opt, r_opt, 0.99)
#         P_val   = np.array(P_val)
#         rho_val = np.array(rho_val)
#         B_prev  = np.maximum(B_start - P_val, 0)
#         rem_prev = np.maximum(rem_start - rho_val, 0)
#         dist = 0
#         for i in range(M):
#             dist += np.exp(-(rmax - rem_prev[i]))
#         tot_dist += dist
#         if t % 200 == 0:
#             print('obj', t, tot_dist/((t+1)))

In [12]:
# policy_run(1001,P_opt, q_opt, r_opt)

In [13]:
for j in range(1, 6):
    print(j, (3*3*2*2)**j)

1 36
2 1296
3 46656
4 1679616
5 60466176


In [16]:
1296/3

432.0

In [14]:
for j in range(1, 6):
    print(j, (3*3*2)**j)

1 18
2 324
3 5832
4 104976
5 1889568


In [15]:
for j in range(1, 6):
    print(j, (3*3)**j)

1 9
2 81
3 729
4 6561
5 59049
